# Notebook 05 ” Full Pipeline Demo
## Crisis Alert System Â· BERT + LSTM + LDA Ensemble

**Goal:** Run the complete pipeline end-to-end ” load all three models, score tweets, fire alerts, and visualise the ensemble distribution and alert breakdown.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))


import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.models.ensemble   import CrisisEnsemble, W_BERT, W_LSTM, W_LDA
from src.alerts.alert_engine import AlertEngine
from src.alerts.alert_schema import LEVEL_COLORS

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
print("Imports OK")

Imports OK


## 1. Load All Models

In [2]:
ensemble = CrisisEnsemble.load(
    bert_path = "../outputs/models/bert_v1",
    lda_path  = "../outputs/models/lda_v1",
    lstm_path = "../outputs/models/lstm_v1",
)
print(f"\nWeights: BERT={W_BERT}  LSTM={W_LSTM}  LDA={W_LDA}")

Loading BERT...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading LDA...
Loading LSTM...

Weights: BERT=0.4  LSTM=0.4  LDA=0.2


## 2. Sanity Check ” Hand-crafted Examples

In [3]:
examples = [
    "Massive wildfire destroys thousands of acres, evacuation orders issued",
    "Oil spill near coast threatens marine life, emergency response deployed",
    "Earthquake 6.8 magnitude hits city center, buildings collapsed casualties reported",
    "Toxic chemical leak at industrial plant, residents told to stay indoors",
    "Flash flood warnings across three states, rivers at record levels",
    "I just had the best coffee of my life, absolutely amazing morning",
    "Looking forward to the game tonight, should be a great match",
    "The new season of my favourite show drops this weekend",
]

results_df = ensemble.predict_df(examples)
pd.set_option("display.max_colwidth", 55)
print(results_df[["text","bert_score","lstm_score","lda_score","crisis_probability","alert_level"]].to_string(index=False))

                                                                              text  bert_score  lstm_score  lda_score  crisis_probability alert_level
            Massive wildfire destroys thousands of acres, evacuation orders issued      0.9958         0.5     0.1828              0.6349      MEDIUM
           Oil spill near coast threatens marine life, emergency response deployed      0.9959         0.5     0.2210              0.6426      MEDIUM
Earthquake 6.8 magnitude hits city center, buildings collapsed casualties reported      0.9960         0.5     0.1960              0.6376      MEDIUM
           Toxic chemical leak at industrial plant, residents told to stay indoors      0.9942         0.5     0.2350              0.6447      MEDIUM
                 Flash flood warnings across three states, rivers at record levels      0.9953         0.5     0.2324              0.6446      MEDIUM
                 I just had the best coffee of my life, absolutely amazing morning      0.0369      

## 3. Score Full Disaster Tweets Dataset

In [4]:
df = pd.read_csv("../data/processed/tweets_clean.csv").dropna(subset=["text_clean"])
print(f"Scoring {len(df):,} tweets (LSTM=0.5 ” no timestamps in Disaster Tweets)...")

full_results = ensemble.predict_df(df["text_clean"].tolist())
full_results["true_label"] = df["label"].values

full_results.to_csv("../data/processed/tweets_ensemble_scores.csv", index=False)
print(f"Saved tweets_ensemble_scores.csv")
print(full_results[["crisis_probability","alert_level","true_label"]].describe().round(3))

Scoring 7,613 tweets (LSTM=0.5 ” no timestamps in Disaster Tweets)...
Saved tweets_ensemble_scores.csv
       crisis_probability  true_label
count            7613.000    7613.000
mean                0.429       0.430
std                 0.170       0.495
min                 0.221       0.000
25%                 0.276       0.000
50%                 0.333       0.000
75%                 0.630       1.000
max                 0.662       1.000


## 4. Alert Level Distribution

In [5]:
from src.models.ensemble import THRESHOLD_CRITICAL, THRESHOLD_HIGH, THRESHOLD_MEDIUM

level_order  = ["LOW","MEDIUM","HIGH","CRITICAL"]
level_counts = full_results["alert_level"].value_counts().reindex(level_order, fill_value=0)
colors       = [LEVEL_COLORS[l] for l in level_order]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar: alert level counts
axes[0].bar(level_order, level_counts.values, color=colors, alpha=0.88, edgecolor="white")
for i, v in enumerate(level_counts.values):
    axes[0].text(i, v + 20, f"{v:,}\n({v/len(full_results)*100:.1f}%)", ha="center", fontsize=10)
axes[0].set_ylabel("Tweet count")
axes[0].set_title("Alert Level Distribution")
axes[0].set_ylim(0, level_counts.max() * 1.18)

# Histogram: crisis probability coloured by alert level
bins = np.linspace(0, 1, 50)
for level, color in zip(level_order, colors):
    subset = full_results[full_results["alert_level"] == level]["crisis_probability"]
    axes[1].hist(subset, bins=bins, color=color, alpha=0.75, label=level)
for thr, lbl in [(THRESHOLD_MEDIUM, "MEDIUM"), (THRESHOLD_HIGH, "HIGH"), (THRESHOLD_CRITICAL, "CRITICAL")]:
    axes[1].axvline(thr, color="black", linestyle="--", lw=1, alpha=0.6)
    axes[1].text(thr + 0.01, axes[1].get_ylim()[1] * 0.85, lbl, fontsize=8, rotation=90, alpha=0.7)
axes[1].set_xlabel("Crisis Probability")
axes[1].set_ylabel("Count")
axes[1].set_title("Score Distribution by Alert Level")
axes[1].legend()

plt.suptitle("Ensemble Alert Distribution ” Disaster Tweets (7,613)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/charts/14_alert_distribution.png", bbox_inches="tight")
plt.show()
print(level_counts)

alert_level
LOW         4591
MEDIUM      3022
HIGH           0
CRITICAL       0
Name: count, dtype: int64


C:\Users\Nafis\AppData\Local\Temp\ipykernel_31300\2747559321.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Ensemble vs Ground Truth

In [6]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

y_true  = full_results["true_label"].astype(int)
y_prob  = full_results["crisis_probability"]
y_pred  = (y_prob >= 0.5).astype(int)

print(f"Ensemble ROC-AUC : {roc_auc_score(y_true, y_prob):.3f}")
print(classification_report(y_true, y_pred, target_names=["Normal","Crisis"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Normal","Crisis"]).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Ensemble Confusion Matrix")

# Score distribution by true label
scores_arr = y_prob.values; labels_arr = y_true.values
axes[1].hist(scores_arr[labels_arr==0], bins=40, alpha=0.7, color="#457b9d", label="Normal", density=True)
axes[1].hist(scores_arr[labels_arr==1], bins=40, alpha=0.7, color="#e63946", label="Crisis",  density=True)
axes[1].axvline(0.5, color="black", linestyle="--", lw=1.2, label="threshold=0.5")
axes[1].set_xlabel("Ensemble crisis probability"); axes[1].set_ylabel("Density")
axes[1].set_title("Ensemble Score by True Label"); axes[1].legend()

plt.suptitle("Ensemble Performance vs Ground Truth", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/charts/15_ensemble_validation.png", bbox_inches="tight")
plt.show()

Ensemble ROC-AUC : 0.963
              precision    recall  f1-score   support

      Normal       0.91      0.96      0.94      4342
      Crisis       0.95      0.87      0.91      3271

    accuracy                           0.92      7613
   macro avg       0.93      0.92      0.92      7613
weighted avg       0.93      0.92      0.92      7613



C:\Users\Nafis\AppData\Local\Temp\ipykernel_31300\673982837.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Fire the Alert Engine ” Top Critical Alerts

In [7]:
engine = AlertEngine(min_level="HIGH")
alerts = engine.process(full_results)
print(f"Alerts fired: {len(alerts)}  (HIGH + CRITICAL only)")

summary = engine.summary_table(alerts)
print(summary.head(10).to_string(index=False))

# Save top 5 alerts as JSON
for alert in sorted(alerts, key=lambda a: a.crisis_probability, reverse=True)[:5]:
    path = engine.save(alert, "../outputs/alerts/")
    print(f"Saved {path}")

Alerts fired: 0  (HIGH + CRITICAL only)
Empty DataFrame
Columns: []
Index: []


## 7. Model Contribution Breakdown

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

model_cols  = ["bert_score", "lstm_score", "lda_score"]
model_names = [f"BERT ({int(W_BERT*100)}%)", f"LSTM ({int(W_LSTM*100)}%)", f"LDA ({int(W_LDA*100)}%)"]
colors_m    = ["#e63946", "#457b9d", "#2a9d8f"]

for ax, col, name, color in zip(axes, model_cols, model_names, colors_m):
    crisis_scores = full_results[full_results["true_label"]==1][col]
    normal_scores = full_results[full_results["true_label"]==0][col]
    ax.hist(normal_scores, bins=40, alpha=0.65, color="#adb5bd", label="Normal", density=True)
    ax.hist(crisis_scores, bins=40, alpha=0.65, color=color,    label="Crisis",  density=True)
    ax.set_title(name); ax.set_xlabel("Score"); ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    mu_c = crisis_scores.mean(); mu_n = normal_scores.mean()
    ax.axvline(mu_c, color=color,    linestyle="--", lw=1.2, label=f"crisis mean={mu_c:.2f}")
    ax.axvline(mu_n, color="#adb5bd", linestyle="--", lw=1.2, label=f"normal mean={mu_n:.2f}")

plt.suptitle("Per-model Score Distributions by True Label", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/charts/16_model_contributions.png", bbox_inches="tight")
plt.show()

# Print mean scores by label
print(full_results.groupby("true_label")[["bert_score","lstm_score","lda_score","crisis_probability"]].mean().round(3))

            bert_score  lstm_score  lda_score  crisis_probability
true_label                                                       
0                0.158         0.5      0.188               0.301
1                0.889         0.5      0.214               0.599


C:\Users\Nafis\AppData\Local\Temp\ipykernel_31300\3716146089.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


: 

## 8. Pipeline Summary

| Component | Dataset | Method | ROC-AUC | Weight |
|---|---|---|---|---|
| BERT | Disaster Tweets (7.6K) | DistilBERT fine-tuned | ~0.91 | 40% |
| LSTM | Climate Twitter (15.8M) | 2-layer LSTM, sentiment velocity | 0.856 | 40% |
| LDA | Disaster Tweets (7.6K) | Gensim LDA k=5 | 0.605 | 20% |
| **Ensemble** | ” | Weighted sum | **see above** | ” |

**Alert thresholds:**
- CRITICAL > 0.85 ” Immediate escalation
- HIGH > 0.70 ” Urgent review
- MEDIUM > 0.50 ” Monitor closely
- LOW â‰¤ 0.50 ” No action